# Assignment III – Generative modeling

Welcome to assignment 3! In this notebook we will help with some parts of the code and the rest is for you to fill in. If you find any errors, or have any feedback, please contact `joelwall@kth.se` or `mats.persson@mi.physics.kth.se`.

In this assignment, to learn about statistical generative models, we will train a flow model to generate samples from a toy distribution in two dimensions. Then, we will move on to generating images from a data set of your choice also using a flow model. There are five image data sets to choose from, namely the same as you encountered in assignment 2:
1. `histopathology` – microscope images of lung tissue, RGB,
2. `galaxies` – telescope images of galaxies, RGB,
3. `brain-stroke` – head CT scans, grayscale,
4. `gravity-wave` – time-frequency spectrograms of LIGO detector glitches, grayscale,
5. `lhc-jets` – simulated particle jets from LHC collisions, grayscale.

In this assignment we use versions of the data sets that have been resampled to $32 \times 32$ pixels. Each data set folder contains a `README` with more information about the images. Note that `gravity-wave` is smaller than the others (624 training images instead of 2100), and that the `lhc-jets` images are very sparse (most pixels are zero).

To generate images we will need to define a special type of CNN called the UNet which has found great success for image tasks in a variety of different fields. If you have time and aim for a higher grade, you will in exercise 5 be asked to extend your image flow model to a diffusion model.

**Structure of the notebook.** The notebook follows the exercise sheet: each section below corresponds to one exercise, and each subsection to one subtask, so you can work through the notebook from top to bottom. Code that is needed but not central to the concepts of the assignment (base classes, data loading, the UNet scaffolding, the training loop and all plotting functions) lives in the file `assignment3_utils.py`. You do not need to change anything there, but feel free to have a look!

**Report.** Hand in a report (PDF) together with this notebook. The report is what is graded, so it must contain the derivations, code excerpts, figures and answers that the exercise sheet asks for; we will not run or read through the notebook to find them.

**Setup.** Make sure that `assignment3_utils.py` is in the same folder as this notebook, and that your data set (the $32 \times 32$ version provided for this assignment) is in the folder `data/` (e.g. `data/brain-stroke/train/...`). If you run the notebook on Google Colab, upload `assignment3_utils.py` and the data folder to the session first.

In [ ]:
import os, sys
import zipfile, urllib.request
from typing import List, Type

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
url = "https://github.com/luk0-00/MLIP-26/releases/download/v2/"
file_to_unzip = "data.zip"
other_file = "assignment3_utils.py"
data_dir = "data"

def download(url, dest):
    with tqdm(unit="B", unit_scale=True, unit_divisor=1024, desc="Downloading") as bar:
        def hook(blocks, block_size, total_size):
            if total_size > 0:
                bar.total = total_size
            bar.update(blocks * block_size - bar.n)
        urllib.request.urlretrieve(url, dest, reporthook=hook)

# Utils file goes next to the notebook, so a plain import works (and the editor can find it)
if not os.path.exists(other_file):
    download(url + other_file, other_file)
else:
    print("Utils already present.")

if not os.path.exists(data_dir):
    download(url + file_to_unzip, file_to_unzip)
    with zipfile.ZipFile(file_to_unzip) as z:
        for member in tqdm(z.infolist(), desc="Extracting", unit="file"):
            z.extract(member, ".")   # the zip already contains the data/ folder
    os.remove(file_to_unzip)
else:
    print("Dataset already present.")


In [ ]:
from assignment3_utils import ( # pyright: ignore[reportMissingImports]
    # Base classes
    SDE, Alpha, Beta,
    # Distributions
    IsotropicGaussian, GaussianMixture, ImageSampler,
    # UNet scaffolding
    UNet,
    # Plotting
    plot_trajectories_1d, plot_densities_2d, plot_corruption_2d, plot_schedulers, plot_loss,
    plot_flow_2d, show_images, plot_corruption_images,
)

# 1. Stochastic differential equations

We will work with SDEs of the form
$$ dX_t = u_t(X_t)dt + \sigma_t dW_t, $$
where $u_t(X_t)$ is the drift coefficient (or vector field) and $\sigma_t$ is the diffusion coefficient. In the code, an SDE is a subclass of the base class `SDE` (defined in `assignment3_utils.py`) which has two methods, `drift_coefficient(xt, t)` and `diffusion_coefficient(xt, t)`, returning $u_t(x_t)$ and $\sigma_t$, respectively.

Exercise 1(a) is a pen-and-paper exercise.

## 1(b) The Euler–Maruyama method
The Euler–Maruyama method is an SDE integrator that, similar to the Euler method for ODEs, is the simplest solver for SDEs:
$$ X_{t+h} = X_t + hu_t(X_t) + \sqrt{h}\sigma_t \epsilon_t,\quad \epsilon_t \sim \mathcal{N}(0,I).$$

Implement a single step in the method `step`. The methods `simulate(x0, ts)` (returns the final state) and `simulate_with_trajectory(x0, ts)` (returns the whole trajectory) are already implemented and call `step` repeatedly along the time discretization `ts`.

Hint: `torch.randn_like(x)` returns samples from a standard normal in a tensor with the same shape as `x`.

In [ ]:
class EulerMaruyamaMethod:
    def __init__(self, sde: SDE):
        self.sde = sde

    def step(self, xt: torch.Tensor, t: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """
        Takes one simulation step
        Args:
            - xt: state at time t, shape (batch_size, ...)
            - t: time, shape (batch_size, 1, ...)
            - h: step size, shape (batch_size, 1, ...)
        Returns:
            - nxt: state at time t + h, shape (batch_size, ...)
        """
        raise NotImplementedError("To be implemented!")

    @torch.no_grad()
    def simulate(self, x: torch.Tensor, ts: torch.Tensor) -> torch.Tensor:
        """
        Simulates the SDE along the time discretization given by ts
        Args:
            - x: initial state at time ts[:, 0], shape (batch_size, ...)
            - ts: time steps, shape (batch_size, num_timesteps, 1, ...)
        Returns:
            - x_final: final state at time ts[:, -1], shape (batch_size, ...)
        """
        nts = ts.shape[1]
        for t_idx in tqdm(range(nts - 1)):
            t = ts[:, t_idx]
            h = ts[:, t_idx + 1] - ts[:, t_idx]
            x = self.step(x, t, h)
        return x

    @torch.no_grad()
    def simulate_with_trajectory(self, x: torch.Tensor, ts: torch.Tensor) -> torch.Tensor:
        """
        Simulates the SDE along the time discretization given by ts and returns the whole trajectory
        Args:
            - x: initial state at time ts[:, 0], shape (batch_size, dim)
            - ts: time steps, shape (batch_size, num_timesteps, 1)
        Returns:
            - xs: trajectory, shape (batch_size, num_timesteps, dim)
        """
        xs = [x.clone()]
        nts = ts.shape[1]
        for t_idx in tqdm(range(nts - 1)):
            t = ts[:, t_idx]
            h = ts[:, t_idx + 1] - ts[:, t_idx]
            x = self.step(x, t, h)
            xs.append(x.clone())
        return torch.stack(xs, dim=1)

## 1(c) Brownian motion
Implement an SDE corresponding to Brownian motion, $u_t = 0$, $\sigma_t = \sigma$.

Hint: `torch.zeros_like` and `torch.ones_like` might come in handy!

In [ ]:
class BrownianMotion(SDE):
    def __init__(self, sigma: float):
        self.sigma = sigma

    def drift_coefficient(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - xt: state at time t, shape (bs, 1)
            - t: time, shape (bs, 1)
        Returns:
            - drift: shape (bs, 1)
        """
        raise NotImplementedError("To be implemented!")

    def diffusion_coefficient(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - xt: state at time t, shape (bs, 1)
            - t: time, shape (bs, 1)
        Returns:
            - diffusion: shape (bs, 1)
        """
        raise NotImplementedError("To be implemented!")

In [ ]:
sigmas = # TODO: choose a few values of sigma, as a list

num_trajectories = 10
x0 = torch.zeros(num_trajectories, 1, device=device)  # initial values - let's start at zero
ts = torch.linspace(0.0, 1.0, 500, device=device).view(1, -1, 1).expand(num_trajectories, -1, 1)  # (num_trajectories, num_timesteps, 1)

fig, axes = plt.subplots(1, len(sigmas), figsize=(6 * len(sigmas), 4), sharey=True, squeeze=False)
for ax, sigma in zip(axes[0], sigmas):
    simulator = EulerMaruyamaMethod(sde=BrownianMotion(sigma))
    trajectories = simulator.simulate_with_trajectory(x0, ts)  # (num_trajectories, num_timesteps, 1)
    plot_trajectories_1d(trajectories, ts, ax=ax, title=rf'Brownian motion with $\sigma={sigma}$')
plt.show()

## 1(d) The Ornstein–Uhlenbeck process
Implement an SDE corresponding to the Ornstein–Uhlenbeck process, $u_t(X_t) = -\theta X_t$, $\sigma_t = \sigma$.

In [ ]:
class OUProcess(SDE):
    def __init__(self, theta: float, sigma: float):
        self.theta = theta
        self.sigma = sigma

    def drift_coefficient(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - xt: state at time t, shape (bs, 1)
            - t: time, shape (bs, 1)
        Returns:
            - drift: shape (bs, 1)
        """
        raise NotImplementedError("To be implemented!")

    def diffusion_coefficient(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - xt: state at time t, shape (bs, 1)
            - t: time, shape (bs, 1)
        Returns:
            - diffusion: shape (bs, 1)
        """
        raise NotImplementedError("To be implemented!")

In [ ]:
# Try comparing multiple choices of (theta, sigma) side-by-side
thetas_and_sigmas = [
    (0.25, 0.0),
    (0.25, 0.25),
    (0.25, 0.5),
]
simulation_time = 20
num_trajectories = 10

fig, axes = plt.subplots(1, len(thetas_and_sigmas), figsize=(7 * len(thetas_and_sigmas), 4))
for ax, (theta, sigma) in zip(axes, thetas_and_sigmas):
    simulator = EulerMaruyamaMethod(sde=OUProcess(theta, sigma))

    x0 = torch.linspace(-10.0, 10.0, num_trajectories, device=device).view(-1, 1)  # a range of initial values
    ts = torch.linspace(0.0, simulation_time, 1000, device=device).view(1, -1, 1).expand(num_trajectories, -1, 1)
    trajectories = simulator.simulate_with_trajectory(x0, ts)

    plot_trajectories_1d(trajectories, ts, ax=ax,
                         title=rf'OU process with $\theta={theta}$, $\sigma={sigma}$, $\sigma^2/2\theta={sigma**2 / (2 * theta):.3f}$')
plt.show()

# 2. The conditional probability path

A data point $z \sim p_{\text{data}}$ is corrupted according to the conditional path
$$ x_t = \alpha_t z + \beta_t \epsilon, \quad \epsilon \sim \mathcal{N}(0, I), $$
where the noise schedulers $\alpha_t, \beta_t$ satisfy $\alpha_0 = \beta_1 = 0$ and $\alpha_1 = \beta_0 = 1$. At $t=0$ we have pure noise and at $t=1$ we have data.

## 2(a) Noise schedulers
The noise schedulers are subclasses of the base classes `Alpha` and `Beta` (defined in `assignment3_utils.py`). They need a `__call__(t)` method that evaluates $\alpha_t$ (or $\beta_t$), and a `dt(t)` method that evaluates the time derivative $\dot\alpha_t$ (or $\dot\beta_t$). The base classes automatically check the boundary conditions when the scheduler is created. If you do not implement `dt`, the base class computes the derivative for you with automatic differentiation.

Implement linear schedulers $\alpha_t = t$, $\beta_t = 1-t$, including their derivatives. Again, `torch.ones_like` might come in handy.

In [ ]:
class LinearAlpha(Alpha):
    """
    Implements alpha_t = t
    """
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: time, shape (num_samples, 1, ...)
        Returns:
            - alpha_t, shape (num_samples, 1, ...)
        """
        raise NotImplementedError("To be implemented!")

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: time, shape (num_samples, 1, ...)
        Returns:
            - d/dt alpha_t, shape (num_samples, 1, ...)
        """
        raise NotImplementedError("To be implemented!")

class LinearBeta(Beta):
    """
    Implements beta_t = 1 - t
    """
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: time, shape (num_samples, 1, ...)
        Returns:
            - beta_t, shape (num_samples, 1, ...)
        """
        raise NotImplementedError("To be implemented!")

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: time, shape (num_samples, 1, ...)
        Returns:
            - d/dt beta_t, shape (num_samples, 1, ...)
        """
        raise NotImplementedError("To be implemented!")

In [ ]:
# Check your implementation: plot alpha_t, beta_t and their derivatives
plot_schedulers({'linear': (LinearAlpha(), LinearBeta())}, derivatives=True)

## 2(b) The conditional path
Here we define a class `ConditionalPath` which holds the data distribution $p_{\text{data}}$, the noise distribution $p_{\text{noise}} = \mathcal{N}(0, I)$ and the noise schedulers. Implement `sample_xt`, which corrupts the data points `z` according to the conditional path. (In exercises 3 and 5 we will extend this class with the conditional vector field and the conditional score.)

In [ ]:
class ConditionalPath(nn.Module):
    def __init__(self, p_data, alpha: Alpha, beta: Beta):
        super().__init__()
        self.p_data = p_data
        self.p_noise = IsotropicGaussian(p_data.dim, std=1.0)
        self.alpha = alpha
        self.beta = beta

    def sample_data(self, num_samples: int) -> torch.Tensor:
        """
        Samples data points z ~ p_data
        Args:
            - num_samples: the number of samples
        Returns:
            - z: samples from p_data, shape (num_samples, ...)
        """
        return self.p_data.sample(num_samples)

    def sample_xt(self, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Samples from the conditional path p_t(x|z) = N(alpha_t * z, beta_t**2 * I)
        Args:
            - z: data points, shape (num_samples, ...)
            - t: time, shape (num_samples, 1, ...)
        Returns:
            - xt: corrupted data points, shape (num_samples, ...)
        """
        alpha_t = self.alpha(t)       # (num_samples, 1, ...)
        beta_t = self.beta(t)         # (num_samples, 1, ...)
        eps = torch.randn_like(z)     # (num_samples, ...)

        raise NotImplementedError("To be implemented!")

## 2(c) A 2D toy distribution
Define your toy data distribution $p_{\text{data}}$. You can for example use a Gaussian mixture with modes evenly placed on a circle, `GaussianMixture.symmetric_2D(nmodes, std, scale)`, or at random positions, `GaussianMixture.random_2D(nmodes, std, scale, seed)`.

In [ ]:
p_data = # TODO: define your toy data distribution
p_data = p_data.to(device)
p_noise = IsotropicGaussian(shape=[2], std=1.0).to(device)

plot_densities_2d(p_noise, p_data, scale=15)

Now simulate the corruption process on 1000 samples from $p_{\text{data}}$ for $t = 0, 0.25, 0.5, 0.75, 1$.

In [ ]:
path = ConditionalPath(p_data=p_data, alpha=LinearAlpha(), beta=LinearBeta()).to(device)

num_samples = 1000
ts = [0.0, 0.25, 0.5, 0.75, 1.0]

z = # TODO: sample num_samples data points from the path

xts = []
for t in ts:
    tt = torch.full((num_samples, 1), t, device=device)  # (num_samples, 1)
    xt = # TODO: corrupt z according to the conditional path at time tt
    xts.append(xt)

plot_corruption_2d(xts, ts, scale=15, title='Linear schedulers')  # use kind='scatter' for scatter plots instead

## 2(d) Non-linear noise schedulers
Implement non-linear noise schedulers that still satisfy the requirements above. You may implement `dt` yourself or leave it out, in which case the derivative is computed with automatic differentiation (then you must use `torch` functions in `__call__`). Then corrupt the *same* data points `z` as in 2(c) with the new schedulers.

In [ ]:
class NonLinearAlpha(Alpha):
    """
    TODO: write down your choice of alpha_t here
    """
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError("To be implemented!")

class NonLinearBeta(Beta):
    """
    TODO: write down your choice of beta_t here
    """
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError("To be implemented!")


plot_schedulers({
    'linear': (LinearAlpha(), LinearBeta()),
    'non-linear': (NonLinearAlpha(), NonLinearBeta()),
})

In [ ]:
path_nonlinear = ConditionalPath(p_data=p_data, alpha=NonLinearAlpha(), beta=NonLinearBeta()).to(device)

xts_nonlinear = []
for t in ts:
    tt = torch.full((num_samples, 1), t, device=device)  # (num_samples, 1)
    xt = # TODO: corrupt the same z as in 2(c) according to the non-linear path at time tt
    xts_nonlinear.append(xt)

plot_corruption_2d(xts_nonlinear, ts, scale=15, title='Non-linear schedulers')

# 3. Flow matching

We will train a vector field $u_t^\theta$ by minimizing the conditional flow matching loss
$$ L(\theta) = \mathbb{E}_{t\sim U[0,1], z\sim p_{\text{data}}, x\sim p_t(\cdot|z)}\left[\left\|u_t^\theta(x) - u_t(x|z)\right\|^2\right]. $$

## 3(a) The conditional vector field
The conditional vector field is given by
$$ u_t(x|z) = \left(\dot\alpha_t - \frac{\dot\beta_t}{\beta_t}\alpha_t\right)z + \frac{\dot\beta_t}{\beta_t}x. $$
We extend `ConditionalPath` from 2(b) with a method that evaluates it. `FlowPath` inherits everything from `ConditionalPath` (e.g. `sample_data` and `sample_xt`), so you only need to implement the new method.

In [ ]:
class FlowPath(ConditionalPath):
    def conditional_vector_field(self, x: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the conditional vector field u_t(x|z)
        Args:
            - x: position variable, shape (num_samples, ...)
            - z: data points, shape (num_samples, ...)
            - t: time, shape (num_samples, 1, ...)
        Returns:
            - conditional vector field, shape (num_samples, ...)
        """
        alpha_t = self.alpha(t)        # (num_samples, 1, ...)
        beta_t = self.beta(t)          # (num_samples, 1, ...)
        dt_alpha_t = self.alpha.dt(t)  # (num_samples, 1, ...)
        dt_beta_t = self.beta.dt(t)    # (num_samples, 1, ...)

        raise NotImplementedError("To be implemented!")

## 3(b) MLP
Now we come to the network architecture. Implement an MLP that takes $(x, t) \in \mathbb{R}^3$ as input and outputs $u_t^\theta(x) \in \mathbb{R}^2$. Given the input dimension `dim` and a list of hidden dimensions `hiddens`, the output should have dimension `dim - 1`, where we subtract 1 since the time variable in the input is one-dimensional.

In [ ]:
class MLP(nn.Module):
    """
    MLP parameterization of the vector field u_t^theta(x)
    """
    def __init__(self, dim: int, hiddens: List[int], activation: Type[nn.Module] = nn.SiLU):
        super().__init__()
        self.dim = dim
        self.net = # TODO: implement the architecture using hiddens (list of hidden dimensions), and apply activation() after each layer except the last layer

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - x: (bs, dim - 1)
            - t: (bs, 1)
        Returns:
            - u_t^theta(x): (bs, dim - 1)
        """
        xt = torch.cat([x, t], dim=-1)  # (bs, dim)
        return self.net(xt)

## 3(c) Training loop
Implement `get_train_loss` according to Algorithm 1: it should sample data from `self.path` and compute the loss between the model output and the conditional vector field. The method `train` takes care of the optimizer and the gradient steps, and returns the list of losses.

In [ ]:
class FlowTrainer:
    def __init__(self, path: FlowPath, model: nn.Module):
        self.path = path
        self.model = model

    def get_train_loss(self, batch_size: int) -> torch.Tensor:
        z = # TODO: sample data from self.path
        t = torch.rand(batch_size, *[1] * (z.ndim - 1), device=z.device)  # (bs, 1) for 2D data, (bs, 1, 1, 1) for images
        x = # TODO: sample corrupted data from self.path

        ut_theta = self.model(x, t)
        ut_ref = # TODO: evaluate the conditional vector field

        error = torch.sum(torch.square(ut_theta - ut_ref).flatten(start_dim=1), dim=-1)  # squared norm, shape (bs,)
        return torch.mean(error)

    def get_optimizer(self, lr: float):
        return torch.optim.Adam(self.model.parameters(), lr=lr)

    def train(self, num_epochs: int, device: torch.device, lr: float = 1e-3, batch_size: int = 1000) -> List[float]:
        # Start
        self.model.to(device)
        self.path.to(device)
        opt = self.get_optimizer(lr)
        self.model.train()

        # Train loop
        losses = []
        pbar = tqdm(range(num_epochs))
        for epoch in pbar:
            opt.zero_grad()
            loss = self.get_train_loss(batch_size)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)  # guards against rare loss spikes that can ruin training
            opt.step()
            losses.append(loss.item())
            pbar.set_description(f'Epoch {epoch}, loss: {loss.item():.4f}')

        # Finish
        self.model.eval()
        return losses

In [ ]:
# Probability path
path = FlowPath(
    p_data = p_data,
    alpha = , # TODO: choose your alpha scheduler
    beta =    # TODO: choose your beta scheduler
).to(device)

# Learnable vector field
mlp = MLP(dim=, hiddens=) # TODO: define the input dimension and hidden layer dimensions

# Train
trainer = FlowTrainer(path, mlp)
losses = trainer.train(num_epochs=1000, device=device, lr=1e-3, batch_size=1000)
plot_loss(losses)

## 3(d) Sampling from the learned flow
To sample new data points we simulate the flow ODE $dX_t = u_t^\theta(X_t)dt$ from $t=0$ to $t=1$, starting from noise $X_0 \sim p_{\text{noise}}$. To reuse our SDE solver, we wrap the trained network in an SDE with zero diffusion. The Euler–Maruyama method then reduces to the Euler method.

In [ ]:
class LearnedODE(SDE):
    def __init__(self, net: nn.Module):
        self.net = net

    def drift_coefficient(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.net(x, t)

    def diffusion_coefficient(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return torch.zeros_like(x)

In [ ]:
num_samples = 300
num_timesteps = 1000  # n_t - try changing this!

ode = LearnedODE(mlp)
simulator = EulerMaruyamaMethod(ode)

x0 = path.p_noise.sample(num_samples)  # (num_samples, 2)
ts = torch.linspace(0.0, 1.0, num_timesteps + 1, device=device).view(1, -1, 1).expand(num_samples, -1, 1)  # (num_samples, num_timesteps + 1, 1)
xts = simulator.simulate_with_trajectory(x0, ts)  # (num_samples, num_timesteps + 1, 2)

plot_flow_2d(path, xts, ts, num_marginals=4, scale=15)

# 4. Image generation

We now move on to images. `ImageSampler` (defined in `assignment3_utils.py`) loads the training images of your data set ($32 \times 32$ pixels, grayscale or RGB depending on the data set) and normalizes the pixel values to $[-1, 1]$. It is used in the same way as the toy distribution, so we can reuse all the code from the previous exercises.

## 4(a) Corrupting images

In [ ]:
p_data = ImageSampler(dataset_path=os.path.join('data', 'galaxies')).to(device) # TODO: choose your image data set
print(f'Loaded {len(p_data.dataset)} images of shape {tuple(p_data.dim)}')

path = FlowPath(p_data=p_data, alpha=LinearAlpha(), beta=LinearBeta()).to(device)

num_images = 9
ts = [0.0, 0.25, 0.5, 0.75, 1.0]

z = # TODO: sample num_images images from the path

xts = []
for t in ts:
    tt = torch.full((num_images, 1, 1, 1), t, device=device)  # (num_images, 1, 1, 1)
    xt = # TODO: corrupt z according to the conditional path at time tt
    xts.append(xt)

plot_corruption_images(xts, ts, nrow=3)

## 4(b) The UNet
The `UNet` class in `assignment3_utils.py` implements the architecture in Figure 1 of the exercise sheet: an initial convolution, encoders, a bottleneck, decoders and a final convolution, together with a Fourier embedding of the time $t$. Every encoder, decoder and the bottleneck consist of residual layers, which it is your job to implement. Follow the right panel of Figure 1:
1. `block1`: SiLU + BatchNorm + Conv (the number of channels and the image size should not change),
2. add the time embedding (`time_adapter` converts it to one value per channel),
3. `block2`: SiLU + BatchNorm + Conv,
4. add the residual connection.

Your `ResidualLayer` class is passed to the `UNet` when it is created.

In [ ]:
class ResidualLayer(nn.Module):
    def __init__(self, channels: int, time_embed_dim: int):
        super().__init__()
        self.block1 = nn.Sequential(
            # TODO: define the first block
        )

        # Converts (bs, time_embed_dim) -> (bs, channels)
        self.time_adapter = nn.Sequential(
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, channels)
        )

        self.block2 = nn.Sequential(
            # TODO: define the second block
        )

    def forward(self, x: torch.Tensor, t_embed: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - x: (bs, c, h, w)
            - t_embed: (bs, t_embed_dim)
        Returns:
            - (bs, c, h, w)
        """
        res = x.clone()                      # (bs, c, h, w)

        # Initial conv block
        x = # TODO: apply first block to x   # (bs, c, h, w)

        # Add time embedding
        t_embed = self.time_adapter(t_embed).unsqueeze(-1).unsqueeze(-1) # (bs, c, 1, 1)
        x = # TODO: add t_embed to x

        # Second conv block
        x = # TODO: apply second block to x  # (bs, c, h, w)

        # Add back residual
        x = # TODO: add res to x             # (bs, c, h, w)

        return x

Train the UNet vector field on your image data set. Training on a GPU takes a few minutes.

In [ ]:
# Probability path
path = FlowPath(
    p_data = p_data,
    alpha = , # TODO: choose your alpha scheduler
    beta =    # TODO: choose your beta scheduler
).to(device)

# Learnable vector field
unet = UNet(
    input_channels = p_data.num_channels,  # 1 for grayscale, 3 for RGB
    channels = [32, 64, 128],
    num_residual_layers = 2,
    t_embed_dim = 40,
    residual_layer = ResidualLayer,
)

# Train!
trainer = FlowTrainer(path, unet)
losses = trainer.train(num_epochs=5000, device=device, lr=1e-3, batch_size=250)
plot_loss(losses)

## 4(c) Generate new images!
Sample images by solving the flow ODE $dX_t = u_t^\theta(X_t)dt$ with your trained UNet as the drift term.

In [ ]:
num_samples = 9
num_timesteps = 100

ode = LearnedODE(unet)
simulator = EulerMaruyamaMethod(ode)

x0 = # TODO: sample initial noise from path.p_noise, shape (num_samples, c, 32, 32)
ts = torch.linspace(0, 1, num_timesteps, device=device).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1)  # (num_samples, num_timesteps, 1, 1, 1)
x1 = # TODO: use the simulator to simulate the ODE starting in x0

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_images(p_data.sample(num_samples), nrow=3, title='Real images', ax=axes[0])
show_images(x1, nrow=3, title='Generated images', ax=axes[1])
plt.show()

## 4(d) Different noise schedulers
Train a new UNet with different noise schedulers (e.g. the non-linear ones from 2(d)) and compare the sample quality to 4(c).

In [ ]:
path_nonlinear = FlowPath(
    p_data = p_data,
    alpha = NonLinearAlpha(), # TODO: try different schedulers
    beta = NonLinearBeta(),
).to(device)

unet_nonlinear = UNet(
    input_channels = p_data.num_channels,
    channels = [32, 64, 128],
    num_residual_layers = 2,
    t_embed_dim = 40,
    residual_layer = ResidualLayer,
)
losses_nonlinear = FlowTrainer(path_nonlinear, unet_nonlinear).train(num_epochs=5000, device=device, lr=1e-3, batch_size=250)
plot_loss(losses_nonlinear)

# Sample with both models, starting from the same noise
x0 = path.p_noise.sample(num_samples)
x1_linear = EulerMaruyamaMethod(LearnedODE(unet)).simulate(x0, ts)
x1_nonlinear = EulerMaruyamaMethod(LearnedODE(unet_nonlinear)).simulate(x0, ts)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_images(x1_linear, nrow=3, title='Schedulers from 4(b)', ax=axes[0])
show_images(x1_nonlinear, nrow=3, title='New schedulers', ax=axes[1])
plt.show()

# 5. From flow to diffusion

The SDE
$$ dX_t = \left(u_t^\theta(X_t) + \frac{\sigma_t^2}{2}\nabla \log p_t(X_t)\right)dt + \sigma_t dW_t $$
has the same probability path $p_t$ as the flow ODE. To turn our flow model into a diffusion model we therefore need to learn the score function $\nabla \log p_t(x)$ with a network $s_t^\theta$, which can be done by minimizing the conditional score matching loss
$$ L(\theta) = \mathbb{E}_{t\sim U[0,1], z\sim p_{\text{data}}, x\sim p_t(\cdot|z)}\left[\left\|s_t^\theta(x) - \nabla \log p_t(x|z)\right\|^2\right]. $$

## 5(a) Conditional score matching
Derive the conditional score $\nabla \log p_t(x|z)$ and implement it by extending `FlowPath` from 3(a).

In [ ]:
class DiffusionPath(FlowPath):
    def conditional_score(self, x: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the conditional score of p_t(x|z)
        Args:
            - x: position variable, shape (num_samples, ...)
            - z: data points, shape (num_samples, ...)
            - t: time, shape (num_samples, 1, ...)
        Returns:
            - conditional score, shape (num_samples, ...)
        """
        alpha_t = self.alpha(t)  # (num_samples, 1, ...)
        beta_t = self.beta(t)    # (num_samples, 1, ...)

        raise NotImplementedError("To be implemented!")

Implement the conditional score matching loss, analogous to `FlowTrainer` in 3(c). `ScoreTrainer` inherits the training loop from `FlowTrainer`, so you only need to implement `get_train_loss`. Note that $\beta_t \to 0$ as $t \to 1$, so the conditional score blows up close to $t = 1$. We therefore sample $t \sim U[0, t_{\max}]$ with $t_{\max}$ slightly smaller than 1.

In [ ]:
class ScoreTrainer(FlowTrainer):
    def __init__(self, path: DiffusionPath, model: nn.Module, t_max: float = 0.99):
        super().__init__(path, model)
        self.t_max = t_max

    def get_train_loss(self, batch_size: int) -> torch.Tensor:
        z = # TODO: sample data from self.path
        t = self.t_max * torch.rand(batch_size, *[1] * (z.ndim - 1), device=z.device)  # (bs, 1, 1, 1)
        x = # TODO: sample corrupted data from self.path

        st_theta = self.model(x, t)
        st_ref = # TODO: evaluate the conditional score

        error = torch.sum(torch.square(st_theta - st_ref).flatten(start_dim=1), dim=-1)  # squared norm, shape (bs,)
        return torch.mean(error)

Train a score network on the image data. The score network must be trained on the same probability path (i.e. the same schedulers) as the vector field from 4(b).

In [ ]:
diffusion_path = DiffusionPath(p_data=p_data, alpha=path.alpha, beta=path.beta).to(device)

score_unet = UNet(
    input_channels = p_data.num_channels,
    channels = [32, 64, 128],
    num_residual_layers = 2,
    t_embed_dim = 40,
    residual_layer = ResidualLayer,
)

score_trainer = ScoreTrainer(diffusion_path, score_unet)
score_losses = score_trainer.train(num_epochs=5000, device=device, lr=1e-3, batch_size=250)
plot_loss(score_losses, title='Score matching loss')

## 5(b) Sampling with the diffusion model
Implement the SDE above, using the trained vector field $u_t^\theta$ (the UNet from 4(b)) and the score network $s_t^\theta$, with a constant diffusion coefficient $\sigma_t = \sigma$. Then sample with a few different values of $\sigma$. Note that $\sigma = 0$ gives back the flow model.

In [ ]:
class LearnedSDE(SDE):
    def __init__(self, flow_model: nn.Module, score_model: nn.Module, sigma: float):
        self.flow_model = flow_model
        self.score_model = score_model
        self.sigma = sigma

    def drift_coefficient(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError("To be implemented!")

    def diffusion_coefficient(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError("To be implemented!")

In [ ]:
sigmas = [0.0, 0.25, 0.5, 1.0]  # TODO: try different values of sigma
num_samples = 9
num_timesteps = 100

x0 = diffusion_path.p_noise.sample(num_samples)  # use the same initial noise for all sigmas
ts = torch.linspace(0, 1, num_timesteps, device=device).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1)

fig, axes = plt.subplots(1, len(sigmas), figsize=(5 * len(sigmas), 5))
for ax, sigma in zip(axes, sigmas):
    sde = LearnedSDE(unet, score_unet, sigma)
    x1 = EulerMaruyamaMethod(sde).simulate(x0, ts)
    show_images(x1, nrow=3, title=rf'$\sigma = {sigma}$', ax=ax)
plt.show()